# Temperature-driven sense switching for ambiguous words (Deepseel API)

In [ ]:
import os
import re
import time
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from openai import OpenAI

os.environ["DEEPSEEK_API_KEY"] = "<YOUR_API_KEY>" # Your API Key
api_key = os.getenv("DEEPSEEK_API_KEY")
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

In [3]:
MODEL_SAMPLING = "deepseek-chat"
MODEL_CLASSIFY  = "deepseek-chat"

WORDS: Dict[str, Dict[str, str]] = {
    #"ring": {
    #    "A": "related to the sound made by a bell, phone, or similar device "
    #        "(e.g., the phone began to ring, a loud ring at the door, "
    #        "the ring of the telephone, the bell will ring at noon)",
    #    "B": "related to a circular band or loop, especially one worn on a finger "
    #        "(e.g., a gold wedding ring, she wore a ring, a shiny ring on the table)",
    #},
    #"second": {
    #    "A": "related to a unit of time equal to one-sixtieth of a minute, "
    #        "including informal expressions where it means 'a very short moment' "
    #        "(e.g., wait just one second, in a second, just a second, "
    #        "I'll be there in a second, it lasted three seconds)",
    #    "B": "related to the position or rank that comes after the first "
    #        "(e.g., finish in second place, the second book, "
    #        "came second in the race, the second floor)",
    #},
    "right": {
        "A": "meaning correct, accurate, or true "
             "(e.g., that's the right answer, you are right, "
             "is this the right way, make sure it's right)",
        "B": "related to a direction or position on the right-hand side, "
             "opposite of left "
             "(e.g., turn right at the corner, the right side of the road, "
             "move it to the right, she sat on his right)",
    },
    #"left": {
    #    "A": "related to a direction or position on the left-hand side, "
    #         "opposite of right "
    #         "(e.g., turn left at the lights, the left side of the stage, "
    #         "he raised his left hand, on the left of the screen)",
    #    "B": "past tense of 'leave' — to have gone away from a place or person, "
    #         "or to have allowed something to remain "
    #         "(e.g., she left the room, he left his keys on the table, "
    #         "they left early, I left it there by mistake)",
    #},
}

TEMPS               = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
N_PER_TEMP          = 50
PROMPT_TEMPLATE     = 'Write one short, simple English sentence using the word "{word}".'
SLEEP_BETWEEN_CALLS = 0.0
BASE_RESULT_DIR     = "result_deepseek_right"

def call_model_text(
    model: str, prompt: str, temperature: float, max_output_tokens: int = 40
) -> str:
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_output_tokens,
    )
    return (resp.choices[0].message.content or "").strip()


def generate_one_sentence(word: str, temperature: float) -> str:
    prompt = PROMPT_TEMPLATE.format(word=word)
    txt = call_model_text(MODEL_SAMPLING, prompt, temperature=temperature, max_output_tokens=30)
    txt = txt.splitlines()[0].strip().strip('"').strip("'")
    return txt

def classify_grammar(sentence: str, max_retries: int = 3) -> bool:
    """
    Check whether a sentence is grammatically correct and natural-sounding.
    Returns True = fluent, False = problematic.
    """
    prompt = f'''Is the following English sentence grammatically correct and natural-sounding?
Answer ONLY "YES" or "NO".

Sentence: {sentence}'''.strip()

    for _ in range(max_retries):
        out = call_model_text(
            MODEL_CLASSIFY, prompt,
            temperature=0.0,
            max_output_tokens=5
        ).upper()
        if "YES" in out:
            return True
        if "NO" in out:
            return False
        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)
    return True

def classify_sense_multi(
    word: str,
    sentence: str,
    senseA: str,
    senseB: str,
    max_retries: int = 3,
) -> str:
    """
    Classify which sense of the word is used.
    Returns:
        "A" -> sense A
        "B" -> sense B
        "C" -> ambiguous / neither A nor B
        "?" -> parsing failed
    """
    prompt = f'''The word "{word}" appears in the sentence below.
Which meaning does it carry?

(A) {senseA}
(B) {senseB}
(C) Neither A nor B, or the meaning is ambiguous / unclear

Answer ONLY with a single letter: A, B, or C.

Sentence: {sentence}'''.strip()

    for _ in range(max_retries):
        out = call_model_text(
            MODEL_CLASSIFY, prompt,
            temperature=0.0,
            max_output_tokens=5
        ).upper()
        m = re.search(r"\b([ABC])\b", out)
        if m:
            return m.group(1)
        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)
    return "?"

def classify_sense(
    word: str,
    sentence: str,
    senseA: str,
    senseB: str,
    max_retries: int = 3,
) -> str:
    """
    Two-stage classification:
        "A" -> sense A
        "B" -> sense B
        "C" -> ambiguous / neither A nor B
        "D" -> sentence is grammatically incorrect or unnatural
        "?" -> parsing failed
    """
    if not classify_grammar(sentence, max_retries=max_retries):
        return "D"

    return classify_sense_multi(word, sentence, senseA, senseB, max_retries=max_retries)

In [ ]:
RUN_IDS = [1, 2, 3, 4, 5]

if not RUN_IDS:
    first_word = next(iter(WORDS))
    word_dir   = os.path.join(BASE_RESULT_DIR, first_word)
    existing   = [
        d for d in os.listdir(word_dir)
        if os.path.isdir(os.path.join(word_dir, d)) and d.isdigit()
    ] if os.path.exists(word_dir) else []
    next_id = max((int(d) for d in existing), default=0) + 1
    RUN_IDS = [next_id]

print(f"本次实验轮次列表：{RUN_IDS}\n")


def save_results(df: pd.DataFrame, base_dir: str, run_id: int) -> None:
    """按 {base_dir}/{word}/{run_id:02d}/T_{temp}.csv 保存。"""
    run_str = f"{run_id:02d}"
    for word in df["word"].unique():
        df_word = df[df["word"] == word]
        for temp in sorted(df_word["temp"].unique()):
            df_slice = df_word[df_word["temp"] == temp].copy()
            save_dir = os.path.join(base_dir, word, run_str)
            os.makedirs(save_dir, exist_ok=True)
            filepath = os.path.join(save_dir, f"T_{temp:.1f}.csv")
            df_slice.to_csv(filepath, index=False, encoding="utf-8-sig")
            print(f"  {len(df_slice):>4} rows → {filepath}")


for RUN_ID in RUN_IDS:
    print(f" 开始轮次 {RUN_ID:02d}")

    for word, senses in WORDS.items():
        senseA, senseB = senses["A"], senses["B"]

        print(f"\n  [{word}]")
        pbar = tqdm(total=len(TEMPS) * N_PER_TEMP, desc=f"  {word}", leave=True)

        for T in TEMPS:
            temp_records = []  # 每个温度独立一批记录

            for _ in range(N_PER_TEMP):
                sent = generate_one_sentence(word, T)
                if SLEEP_BETWEEN_CALLS:
                    time.sleep(SLEEP_BETWEEN_CALLS)
                lab = classify_sense(word, sent, senseA, senseB)
                if SLEEP_BETWEEN_CALLS:
                    time.sleep(SLEEP_BETWEEN_CALLS)

                temp_records.append({
                    "word":     word,
                    "run":      RUN_ID,
                    "temp":     T,
                    "sentence": sent,
                    "label":    lab,
                })
                pbar.update(1)

            df_temp = pd.DataFrame(temp_records)
            save_results(df_temp, base_dir=BASE_RESULT_DIR, run_id=RUN_ID)
            print(f"   [{word}] T={T:.1f} → {len(df_temp)} 条已保存，轮次 {RUN_ID:02d}")

        pbar.close()

print(f"\n 全部完成：{len(RUN_IDS)} 个轮次，{len(WORDS)} 个词")

本次实验轮次列表：[1, 2, 3, 4, 5]

 开始轮次 01

  [right]


  right:   0%|          | 0/1050 [00:00<?, ?it/s]

    50 rows → result_deepseek_right/right/01/T_0.0.csv
   [right] T=0.0 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_0.1.csv
   [right] T=0.1 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_0.2.csv
   [right] T=0.2 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_0.3.csv
   [right] T=0.3 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_0.4.csv
   [right] T=0.4 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_0.5.csv
   [right] T=0.5 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_0.6.csv
   [right] T=0.6 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_0.7.csv
   [right] T=0.7 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_0.8.csv
   [right] T=0.8 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_0.9.csv
   [right] T=0.9 → 50 条已保存，轮次 01
    50 rows → result_deepseek_right/right/01/T_1.0.csv
   [right] T=1.0 → 50 条已保存，轮次 01
    50 rows → result_deepseek_ri

  right:   0%|          | 0/1050 [00:00<?, ?it/s]

    50 rows → result_deepseek_right/right/02/T_0.0.csv
   [right] T=0.0 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_0.1.csv
   [right] T=0.1 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_0.2.csv
   [right] T=0.2 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_0.3.csv
   [right] T=0.3 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_0.4.csv
   [right] T=0.4 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_0.5.csv
   [right] T=0.5 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_0.6.csv
   [right] T=0.6 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_0.7.csv
   [right] T=0.7 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_0.8.csv
   [right] T=0.8 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_0.9.csv
   [right] T=0.9 → 50 条已保存，轮次 02
    50 rows → result_deepseek_right/right/02/T_1.0.csv
   [right] T=1.0 → 50 条已保存，轮次 02
    50 rows → result_deepseek_ri

...

    50 rows → result_deepseek_right/right/03/T_0.0.csv
   [right] T=0.0 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_0.1.csv
   [right] T=0.1 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_0.2.csv
   [right] T=0.2 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_0.3.csv
   [right] T=0.3 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_0.4.csv
   [right] T=0.4 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_0.5.csv
   [right] T=0.5 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_0.6.csv
   [right] T=0.6 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_0.7.csv
   [right] T=0.7 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_0.8.csv
   [right] T=0.8 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_0.9.csv
   [right] T=0.9 → 50 条已保存，轮次 03
    50 rows → result_deepseek_right/right/03/T_1.0.csv
   [right] T=1.0 → 50 条已保存，轮次 03
    50 rows → result_deepseek_ri

  right:   0%|          | 0/1050 [00:00<?, ?it/s]

    50 rows → result_deepseek_right/right/04/T_0.0.csv
   [right] T=0.0 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_0.1.csv
   [right] T=0.1 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_0.2.csv
   [right] T=0.2 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_0.3.csv
   [right] T=0.3 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_0.4.csv
   [right] T=0.4 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_0.5.csv
   [right] T=0.5 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_0.6.csv
   [right] T=0.6 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_0.7.csv
   [right] T=0.7 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_0.8.csv
   [right] T=0.8 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_0.9.csv
   [right] T=0.9 → 50 条已保存，轮次 04
    50 rows → result_deepseek_right/right/04/T_1.0.csv
   [right] T=1.0 → 50 条已保存，轮次 04
    50 rows → result_deepseek_ri

  right:   0%|          | 0/1050 [00:00<?, ?it/s]

    50 rows → result_deepseek_right/right/05/T_0.0.csv
   [right] T=0.0 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_0.1.csv
   [right] T=0.1 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_0.2.csv
   [right] T=0.2 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_0.3.csv
   [right] T=0.3 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_0.4.csv
   [right] T=0.4 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_0.5.csv
   [right] T=0.5 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_0.6.csv
   [right] T=0.6 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_0.7.csv
   [right] T=0.7 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_0.8.csv
   [right] T=0.8 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_0.9.csv
   [right] T=0.9 → 50 条已保存，轮次 05
    50 rows → result_deepseek_right/right/05/T_1.0.csv
   [right] T=1.0 → 50 条已保存，轮次 05
    50 rows → result_deepseek_ri